# 提交生成 — 三任务全量训练 + 预测test + 组装提交csv
IEEE BigData 2026 Explainable Suicide Risk Detection

用全量train训练最终模型，预测test(leaderboard.xlsx)，按官方格式生成 YourTeamName.csv
- task1a: DepRoBERTa + RoBERTa 融合(权重0.55/0.45) → risk_level
- task2:  DepRoBERTa + 逐类阈值(从results读) → factors
- task1b: DepRoBERTa BIO + indicator规则 → evidence

**运行前**：GPU；Drive里有 common.py + train_clean.csv + leaderboard.xlsx + results/oof_t2_deproberta.npz


## 0. 配置 —— 改这里


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/IEEE_BigData2026"   # <<<< 改路径
TEST_FILE   = "leaderboard.xlsx"        # test文件名(放在PROJECT_DIR下)
TEAM_NAME   = "YourTeamName"            # <<<< 改成你的队名(决定输出文件名)

# 融合权重(task1a，已在OOF上验证)
W_DEP = 0.55   # DepRoBERTa 占比
W_ROB = 0.45   # RoBERTa 占比

# 超参
MAX_LEN=512; BATCH_SIZE=8; LR=2e-5; SEED=42
EPOCHS_1A=4; EPOCHS_T2=5; EPOCHS_1B=4

# 要不要做task1b(若还没确认work可设False，提交里evidence留空)
DO_TASK1B = True


## 1. 挂载+依赖+读数据


In [ ]:
import os, sys
try:
    from google.colab import drive; drive.mount("/content/drive")
except Exception: print("非Colab")
assert os.path.isdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR); os.chdir(PROJECT_DIR)
import importlib, subprocess
def ensure(p,i=None):
    try: importlib.import_module(i or p)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",p])
for p,i in [("sentencepiece","sentencepiece"),("accelerate","accelerate")]:
    ensure(p,i)
import torch, numpy as np, pandas as pd
assert torch.cuda.is_available(), "选GPU！"
print("GPU:", torch.cuda.get_device_name(0))

from common import (load_data, FACTORS, FACTOR_COLS, MODELS, RISK_LEVELS, RISK2ID, ID2RISK)
train_df = load_data()
train_df["label_id"] = train_df["risk_level"].map(RISK2ID)

# 读test
test_df = pd.read_excel(TEST_FILE)
print(f"train {len(train_df)} 行, test {len(test_df)} 行")
print("test列:", list(test_df.columns))

def set_seed(s=SEED):
    import random; random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()


## 2. 通用：Dataset + 全量训练函数


In [ ]:
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForTokenClassification, Trainer, TrainingArguments)

class TextDS(Dataset):
    def __init__(self, texts, labels, tok, max_len, multilabel=False):
        self.texts=list(texts); self.labels=labels; self.tok=tok
        self.max_len=max_len; self.multilabel=multilabel
    def __len__(self): return len(self.texts)
    def __getitem__(self,i):
        enc=self.tok(str(self.texts[i]),truncation=True,max_length=self.max_len,
                     padding="max_length",return_tensors="pt")
        item={k:v.squeeze(0) for k,v in enc.items()}
        if self.labels is not None:
            if self.multilabel:
                item["labels"]=torch.tensor(self.labels[i],dtype=torch.float)
            else:
                item["labels"]=torch.tensor(self.labels[i],dtype=torch.long)
        return item

def train_full(model_name, texts, labels, epochs, num_labels,
               problem_type=None, multilabel=False):
    """用全量数据训一个模型，返回 (model, tokenizer)"""
    tok=AutoTokenizer.from_pretrained(model_name, use_fast=True)
    kwargs={"num_labels":num_labels}
    if problem_type: kwargs["problem_type"]=problem_type
    model=AutoModelForSequenceClassification.from_pretrained(model_name,**kwargs).to("cuda")
    ds=TextDS(texts, labels, tok, MAX_LEN, multilabel)
    use_bf16=torch.cuda.is_bf16_supported()
    args=TrainingArguments(output_dir="/content/tmp_ckpt",num_train_epochs=epochs,
        per_device_train_batch_size=BATCH_SIZE,learning_rate=LR,save_strategy="no",
        logging_steps=100,warmup_ratio=0.1,weight_decay=0.01,
        bf16=use_bf16,fp16=not use_bf16,report_to="none",seed=SEED)
    Trainer(model=model,args=args,train_dataset=ds).train()
    return model, tok

@torch.no_grad()
def predict_probs(model, tok, texts, multilabel=False):
    """预测，返回概率矩阵"""
    model.eval(); out=[]
    for i in range(0,len(texts),32):
        batch=[str(t) for t in texts[i:i+32]]
        enc=tok(batch,truncation=True,max_length=MAX_LEN,padding=True,return_tensors="pt").to("cuda")
        logits=model(**enc).logits
        if multilabel: probs=torch.sigmoid(logits)
        else: probs=torch.softmax(logits,dim=1)
        out.append(probs.cpu().numpy())
    return np.concatenate(out,0)


## 3. Task1a：训DepRoBERTa+RoBERTa → 融合 → 预测test风险


In [ ]:
print("\n===== Task1a: DepRoBERTa =====")
set_seed()
m_dep, t_dep = train_full(MODELS["deproberta"], train_df["post"], train_df["label_id"].values,
                          EPOCHS_1A, num_labels=4)
p_dep_test = predict_probs(m_dep, t_dep, test_df["post"].tolist())
del m_dep; torch.cuda.empty_cache()

print("\n===== Task1a: RoBERTa =====")
set_seed()
m_rob, t_rob = train_full(MODELS["roberta"], train_df["post"], train_df["label_id"].values,
                          EPOCHS_1A, num_labels=4)
p_rob_test = predict_probs(m_rob, t_rob, test_df["post"].tolist())
del m_rob; torch.cuda.empty_cache()

# 融合
p_fused = W_DEP*p_dep_test + W_ROB*p_rob_test
risk_pred = [ID2RISK[i] for i in p_fused.argmax(1)]
print("\ntest风险分布:", pd.Series(risk_pred).value_counts().to_dict())


## 4. Task2：训DepRoBERTa(多标签) → 套用已存阈值 → 预测test因子


In [ ]:
print("\n===== Task2: DepRoBERTa multi-label =====")
Y_train = train_df[FACTOR_COLS].values.astype(np.float32)
set_seed()
m_t2, t_t2 = train_full(MODELS["deproberta"], train_df["post"], Y_train,
                        EPOCHS_T2, num_labels=len(FACTORS),
                        problem_type="multi_label_classification", multilabel=True)
p_t2_test = predict_probs(m_t2, t_t2, test_df["post"].tolist(), multilabel=True)
del m_t2; torch.cuda.empty_cache()

# 读已存的逐类阈值
thr = np.load(os.path.join(PROJECT_DIR,"results","oof_t2_deproberta.npz"),
              allow_pickle=True)["thresholds"]
print("套用逐类阈值(前5):", np.round(thr[:5],2))

# 预测因子：每类用各自阈值
factors_pred=[]
for row in p_t2_test:
    fs=[FACTORS[c] for c in range(len(FACTORS)) if row[c]>thr[c]]
    factors_pred.append(fs)
print("平均每帖因子数:", np.mean([len(f) for f in factors_pred]).round(2))


## 5. Task1b：训DepRoBERTa(BIO) → indicator规则 → 预测test证据


In [ ]:
evidence_pred = [""]*len(test_df)
if DO_TASK1B:
    print("\n===== Task1b: DepRoBERTa BIO =====")
    from common import get_fold  # noqa
    tok_b=AutoTokenizer.from_pretrained(MODELS["deproberta"], use_fast=True)

    def char_mask(post, ev):
        pl=post.lower(); m=[0]*len(post)
        if isinstance(ev,str):
            for sp in ev.split(';'):
                sp=sp.strip()
                if not sp or sp.lower()=='none': continue
                spl=sp.lower(); st=0
                while True:
                    idx=pl.find(spl,st)
                    if idx<0: break
                    for k in range(idx,idx+len(spl)): m[k]=1
                    st=idx+len(spl)
        return m

    class BIODS(Dataset):
        def __init__(self, df_sub):
            self.posts=df_sub["post"].astype(str).tolist()
            self.evs=df_sub["evidence"].astype(str).tolist()
        def __len__(self): return len(self.posts)
        def __getitem__(self,i):
            post=self.posts[i]; cm=char_mask(post,self.evs[i])
            enc=tok_b(post,truncation=True,max_length=MAX_LEN,padding="max_length",
                      return_offsets_mapping=True,return_tensors="pt")
            offs=enc["offset_mapping"][0].tolist(); labels=[]; prev=0
            for (s,e) in offs:
                if s==e: labels.append(-100)
                else:
                    isev=any(cm[s:e]) if e<=len(cm) else False
                    labels.append((1 if prev==0 else 2) if isev else 0)
                    prev=1 if isev else 0
            enc.pop("offset_mapping")
            item={k:v.squeeze(0) for k,v in enc.items()}
            item["labels"]=torch.tensor(labels)
            return item

    set_seed()
    m_b=AutoModelForTokenClassification.from_pretrained(MODELS["deproberta"],num_labels=3).to("cuda")
    use_bf16=torch.cuda.is_bf16_supported()
    args=TrainingArguments(output_dir="/content/tmp_b",num_train_epochs=EPOCHS_1B,
        per_device_train_batch_size=BATCH_SIZE,learning_rate=LR,save_strategy="no",
        logging_steps=100,warmup_ratio=0.1,weight_decay=0.01,
        bf16=use_bf16,fp16=not use_bf16,report_to="none",seed=SEED)
    Trainer(model=m_b,args=args,train_dataset=BIODS(train_df)).train()

    # 预测test证据
    m_b.eval()
    for i,row in test_df.iterrows():
        post=str(row["post"])
        # indicator规则：直接空（但test没有risk标签，改用task1a的预测）
        if risk_pred[i]=="indicator":
            evidence_pred[i]=""; continue
        enc=tok_b(post,truncation=True,max_length=MAX_LEN,padding="max_length",
                  return_offsets_mapping=True,return_tensors="pt")
        offs=enc["offset_mapping"][0].tolist()
        with torch.no_grad():
            inp={k:v.to("cuda") for k,v in enc.items() if k!="offset_mapping"}
            preds=m_b(**inp).logits[0].argmax(-1).cpu().tolist()
        spans=[]; cs=None
        for (s,e),l in zip(offs,preds):
            if s==e: continue
            if l in (1,2):
                if cs is None: cs=s
                ce=e
            else:
                if cs is not None: spans.append(post[cs:ce]); cs=None
        if cs is not None: spans.append(post[cs:ce])
        evidence_pred[i]="; ".join(spans)
    del m_b; torch.cuda.empty_cache()
    print("有证据的test帖数:", sum(1 for e in evidence_pred if e))


## 6. 组装提交csv + 自检


In [ ]:
sub=pd.DataFrame({
    "row_id": test_df["row_id"].values,
    "risk_level": risk_pred,
    "evidence": evidence_pred,
    "factors": [str(f) for f in factors_pred],  # list转字符串
})

# 自检
print("=== 提交前自检 ===")
assert len(sub)==len(test_df), "行数不对"
assert sub["risk_level"].isin(["Indicator","Ideation","Behavior","Attempt",
                                "indicator","ideation","behavior","attempt"]).all(), "风险取值异常"
# 官方要求首字母大写
sub["risk_level"]=sub["risk_level"].str.capitalize()
print("行数:", len(sub), "✓")
print("风险分布:", sub["risk_level"].value_counts().to_dict())
print("有证据帖数:", (sub["evidence"]!="").sum())
print("平均因子数:", np.mean([len(eval(f)) for f in sub["factors"]]).round(2))

out_path=os.path.join(PROJECT_DIR, f"{TEAM_NAME}.csv")
sub.to_csv(out_path, index=False)
print(f"\n✓ 提交文件已生成: {out_path}")
print(sub.head(3).to_string())
